# ARC/ATLAS SynthSR Upsampling

Applies **SynthSR** (Iglesias et al., *Science Advances* 2023) to the `test_lores` set —
198 naturally low-quality MRIs that were held out entirely from v3 training.

SynthSR produces a synthetic 1mm isotropic T1w from any input MRI quality/contrast,
without requiring paired training data.

**Pipeline:**
1. Input: `test_lores/t1/` — ANTs-registered, normalised [0,1] T1s
2. SynthSR enhances each image → synthetic high-quality T1
3. Re-normalise output to [0,1] (v3 format)
4. Copy masks unchanged from `test_lores/masks/` (same subject, same MNI space)
5. Output: `Upsampled_LowRes/t1/` and `Upsampled_LowRes/masks/`

**Research question:** Does SR preprocessing recover the segmentation accuracy lost
due to natural image degradation?

In [1]:
import sys, os, shutil, subprocess
from pathlib import Path
import numpy as np
import nibabel as nib

# --- Paths ---
V4_ROOT    = Path("/home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/ARC_ATLAS_Train_v4")
SPLIT_BASE = Path("/home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/A_A_Combined_Data/Processed_HiresLowres_Split_Data")
SRC_T1_DIR   = SPLIT_BASE / "test_lores/t1"
SRC_MASK_DIR = SPLIT_BASE / "test_lores/masks"
OUT_DIR      = Path("/home/rbielski/stroke_cleaned/ARC_ATLAS_Combined/A_A_Combined_Data/Upsampled_LowRes")
OUT_T1_DIR   = OUT_DIR / "t1"
OUT_MASK_DIR = OUT_DIR / "masks"
TMP_DIR      = OUT_DIR / "_synthsr_tmp"

for d in [OUT_T1_DIR, OUT_MASK_DIR, TMP_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Import normalize_t1 from v4 prep_utils
sys.path.insert(0, str(V4_ROOT / "src" / "data_prep"))
from prep_utils import normalize_t1

OVERWRITE = False
N_JOBS    = 1   # SynthSR runs its own internal parallelism; set >1 only if you have many GPUs

# Sanity checks
src_t1s = sorted(SRC_T1_DIR.glob("*_T1w_MNI_norm.nii.gz"))
assert len(src_t1s) > 0, f"No T1s found in {SRC_T1_DIR}"
print(f"Source T1s: {len(src_t1s)}")
print(f"Source masks: {len(list(SRC_MASK_DIR.glob('*.nii.gz')))}")

Source T1s: 198
Source masks: 198


In [2]:
# Install SynthSR if not already available
import importlib.util

def check_synthsr():
    """Check if mri_synthsr command is available."""
    result = subprocess.run(["which", "mri_synthsr"],
                            capture_output=True, text=True)
    return result.returncode == 0

if not check_synthsr():
    print("SynthSR not found — installing...")
    subprocess.run([sys.executable, "-m", "pip", "install", "SynthSR"], check=True)
    # After install, mri_synthsr should be in the same bin as the python executable
    synthsr_bin = Path(sys.executable).parent / "mri_synthsr"
else:
    synthsr_bin = Path(subprocess.run(["which", "mri_synthsr"],
                                      capture_output=True, text=True).stdout.strip())

assert synthsr_bin.exists(), f"mri_synthsr not found at {synthsr_bin}"
print(f"SynthSR binary: {synthsr_bin}")

# Check GPU availability for SynthSR
try:
    import tensorflow as tf
    gpus = tf.config.list_physical_devices('GPU')
    USE_CPU = len(gpus) == 0
    print(f"GPUs available: {len(gpus)} — {'using CPU' if USE_CPU else 'using GPU'}")
except Exception:
    USE_CPU = True
    print("TensorFlow not available for GPU check — defaulting to CPU")

SynthSR not found — installing...


ERROR: Could not find a version that satisfies the requirement SynthSR (from versions: none)
ERROR: No matching distribution found for SynthSR


CalledProcessError: Command '['/home/rbielski/miniconda3/envs/tf_310/bin/python', '-m', 'pip', 'install', 'SynthSR']' returned non-zero exit status 1.

In [ ]:
# Run SynthSR on each test_lores T1, then normalise output

def run_synthsr(t1_path: Path) -> bool:
    """Run SynthSR on a single T1, normalise, save to OUT_T1_DIR."""
    key = t1_path.name.replace("_T1w_MNI_norm.nii.gz", "")
    out_t1 = OUT_T1_DIR / t1_path.name

    if not OVERWRITE and out_t1.exists():
        return True

    tmp_out = TMP_DIR / f"{key}_synthsr_raw.nii.gz"

    try:
        cmd = [str(synthsr_bin), "--i", str(t1_path), "--o", str(tmp_out)]
        if USE_CPU:
            cmd.append("--cpu")
        result = subprocess.run(cmd, capture_output=True, text=True)
        if result.returncode != 0:
            print(f"[ERROR] SynthSR failed for {key}:\n{result.stderr[-500:]}")
            return False

        # Normalise output to [0,1] (v3 format)
        img  = nib.load(str(tmp_out))
        norm = normalize_t1(img.get_fdata(dtype=np.float32))
        nib.save(nib.Nifti1Image(norm, img.affine, img.header), str(out_t1))

        if tmp_out.exists():
            tmp_out.unlink()
        return True

    except Exception as e:
        print(f"[ERROR] {key}: {e}")
        if tmp_out.exists():
            tmp_out.unlink()
        return False


# Filter to not-yet-done subjects
todo = [p for p in src_t1s
        if OVERWRITE or not (OUT_T1_DIR / p.name).exists()]
print(f"To process: {len(todo)}  (skipping {len(src_t1s) - len(todo)} already done)\n")

ok = fail = 0
for i, t1_path in enumerate(todo):
    key = t1_path.name.replace("_T1w_MNI_norm.nii.gz", "")
    print(f"[{i+1}/{len(todo)}] {key}")
    if run_synthsr(t1_path):
        ok += 1
    else:
        fail += 1

print(f"\nSynthSR done -- processed: {ok}, failed: {fail}, skipped: {len(src_t1s) - len(todo)}")

In [ ]:
# Copy masks from test_lores/masks/ -> Upsampled_LowRes/masks/
# No processing needed: same subject, same MNI registration, SR doesn't move voxels.

src_masks = sorted(SRC_MASK_DIR.glob("*_lesion_mask_MNI_clean.nii.gz"))
print(f"Source masks: {len(src_masks)}")

copied = skipped = 0
for mask_path in src_masks:
    dst = OUT_MASK_DIR / mask_path.name
    # Only copy if the corresponding SR T1 was successfully produced
    key = mask_path.name.replace("_lesion_mask_MNI_clean.nii.gz", "")
    t1_done = (OUT_T1_DIR / f"{key}_T1w_MNI_norm.nii.gz").exists()
    if not t1_done:
        print(f"[skip mask] {key} — SR T1 not found")
        continue
    if not OVERWRITE and dst.exists():
        skipped += 1
        continue
    shutil.copy2(str(mask_path), str(dst))
    copied += 1

print(f"Masks — copied: {copied}, skipped: {skipped}")

In [ ]:
# Verification
print("=" * 50)
print("VERIFICATION")
print("=" * 50)

out_t1s   = list(OUT_T1_DIR.glob("*.nii.gz"))
out_masks = list(OUT_MASK_DIR.glob("*.nii.gz"))
print(f"Output T1s:   {len(out_t1s)}  (expected ~{len(src_t1s)})")
print(f"Output masks: {len(out_masks)}")

# Check a sample: nonzero %, shape
sample = out_t1s[0] if out_t1s else None
if sample:
    img = nib.load(str(sample))
    data = img.get_fdata(dtype=np.float32)
    nz_pct = 100 * np.count_nonzero(data) / data.size
    print(f"\nSample: {sample.name}")
    print(f"  Shape: {img.shape}")
    print(f"  Nonzero: {nz_pct:.1f}%")
    print(f"  Value range: [{data.min():.4f}, {data.max():.4f}]")